# Genotype–Phenotype Heterogeneity Dataset Exploration with `mlcroissant`
This notebook guides you to load, explore, and process the FAIR<sup>2</sup> "Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency" dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Output the dataset metadata
metadata = dataset.metadata.to_json()
print(f"Dataset Name: {metadata.get('name')}")
print(f"Description: {metadata.get('description')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Retrieve all record sets defined by their `@id`s
record_set_ids = dataset.metadata.recordSet

if not record_set_ids:
    # If empty, attempt to infer available tables from mlcroissant
    record_sets = dataset.record_sets()
    record_set_ids = [rs['@id'] for rs in record_sets]
else:
    # Already a list of record set IDs
    record_sets = [dataset.metadata.as_dict_by_id(rsid) for rsid in record_set_ids]

print("Record Sets and their @id:")
for rset in record_sets:
    print(f"- {rset.get('@id')} : {rset.get('name', '(no name)')}")
    fields = rset.get('field', [])
    if fields:
        print("  Fields and their @id:")
        for fld in fields:
            field_info = dataset.metadata.as_dict_by_id(fld) if isinstance(fld, str) else fld
            print(f"    - {field_info.get('@id')} : {field_info.get('name', '(no name)')}")
    else:
        print("  (No fields listed)")
    print("")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for record set @id: {record_set_id} with shape {df.shape}")
        print(f"Columns (@id): {df.columns.tolist()}")
        print(df.head())
    else:
        print(f"No records found for record set @id: {record_set_id}")
print(f"Available DataFrames: {list(dataframes.keys())}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping by key attributes to prepare for further analysis.

> Note: For demonstration, select a numeric field (e.g., age at diagnosis) and a categorical field (e.g., phenotype or presence of infertility) using their `@id`.

In [ ]:
# Choose representative record set and fields by @id
# Suppose record_set_ids[0] is clinical features (Table 1)
record_set_id = record_set_ids[0]
df = dataframes[record_set_id]

# Find the field IDs for numeric and categorical fields

# For demonstration, try to identify column @id for 'Age at diagnosis' and 'Infertility'
field_ids = df.columns.tolist()
print("Fields (@id) in the selected DataFrame:", field_ids)

# Try typical field names based on dataset description
numeric_field_id = None
group_field_id = None
for col in field_ids:
    if 'age' in col.lower():
        numeric_field_id = col
    if 'infertility' in col.lower():
        group_field_id = col

print(f"Selected numeric_field_id: {numeric_field_id}")
print(f"Selected group_field_id: {group_field_id}")

# Filter records with Age > threshold
if numeric_field_id and numeric_field_id in df.columns:
    threshold = 25
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())
    # Normalize numeric field
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id}:")
    print(filtered_df[[numeric_field_id, norm_col]].head())
    # Group data by categorical field, if possible
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())
else:
    print("Could not process numeric field for EDA. Please check field @id mappings.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Plot distribution of age at diagnosis if data is available
if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(6, 4))
    df[numeric_field_id].hist(bins=5)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
    
    # Scatter plot of age vs normalized
    if f"{numeric_field_id}_normalized" in filtered_df.columns:
        plt.figure(figsize=(6,4))
        plt.scatter(filtered_df[numeric_field_id], filtered_df[f"{numeric_field_id}_normalized"], c='blue', label='Patients')
        plt.title("Age at diagnosis vs Normalized Age")
        plt.xlabel(numeric_field_id)
        plt.ylabel(f"{numeric_field_id}_normalized")
        plt.legend()
        plt.show()

# Bar plot for grouped data
if group_field_id and group_field_id in df.columns:
    group_counts = df[group_field_id].value_counts()
    group_counts.plot(kind='bar')
    plt.title(f"Number of patients by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel("Count")
    plt.show()

## 6. Conclusion
This notebook demonstrated how to load, inspect, and process clinical records and genotype–phenotype tabular data from a FAIR2 dataset using the `mlcroissant` library.

**Key findings:**
- The dataset organizes phenotypic, hormonal, imaging, and genetic variant information for three rare-disease cases, indexed by their `@id` identifiers for record sets and fields.
- Exploratory analysis illustrated basic operations such as filtering on age, normalization, and categorical grouping (e.g., infertility presence).
- Visualizations showed the distribution of age and group counts, highlighting the potential for further exploration despite the dataset's small size and rarity.

Refer to the dataset schema and FAIR2 documentation for detailed entity `@id`s for reproducible analysis and cross-dataset interoperability.